In [24]:

import torch
import bisect
import numpy as np
# from numpy.dtypes import StringDType
import csv
from random import randrange
import subprocess
import mmap
import matplotlib.pyplot as plt

In [25]:
import bpe_tokeniser
tokeniser = bpe_tokeniser.Tokeniser("../orthography_cleaning/merge_rules.csv", 8000)

In [26]:
#!pip install colorama
from colorama import Fore, Style
#from google.colab import drive
#drive.mount('/content/drive')

In [27]:
#%cd /content/drive/MyDrive/PyTorch/Tokenisation/padunk/model_versions/tokeniser2000_windowsize96_LSTM+GRUdecoder

In [28]:
batch_size = 32
transformer_window_length = 96
token_vocabulary_size = 8043
transformer_stride = transformer_window_length//2
token_embedding_dim=256
char_vocab_size=43

In [29]:
def countLines (file_path) -> int:
  line_count = 0
  with open(file_path, "r") as python_filehandle:
    with mmap.mmap(python_filehandle.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_filehandle:
        while mmap_filehandle.readline():
            line_count += 1

  return line_count

In [30]:
def removeOOVChars(word):
    cleaned_word = "";
    for char in word:
        if char not in char_dict_reversed:
            cleaned_word += "<unk>"
        else:
            cleaned_word += char

    return cleaned_word

In [31]:
def splitTensorIntoStridedWindows(one_d_tensor, window_length, stride):
    tensor_length = one_d_tensor.numel()
    padding_needed = (stride - (tensor_length - window_length)%stride)%stride

    return torch.nn.functional.pad(one_d_tensor, (0, padding_needed), value=0).unfold(0, window_length, stride)

In [32]:
def getTokenWindows(tokens_tensor, text_tokenoffsets):
    batched_texts = []
    for text_tokens in tokens_tensor.tensor_split(text_tokenoffsets[1:]):
        batched_texts.append(splitTensorIntoStridedWindows(text_tokens, transformer_window_length, transformer_stride))

    return(torch.cat(batched_texts, dim=0))

In [33]:
def getTokenisedWindows(words_list, text_wordoffsets, bpe_dropout_probability=0.1):
    flat_token_ids, word_tokenoffsets = tokeniser.tokenise(words_list, bpe_dropout_probability)
    return getTokenWindows(torch.from_numpy(flat_token_ids).long(), torch.from_numpy(word_tokenoffsets)[text_wordoffsets])
    

In [34]:
def getTokenWindowsDataLoader(token_windows, batch_size, shuffle):
    dataset = tokenWindowsDataset(token_windows)
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [35]:
def stringifyTokensList(tokens_list, tokens_dict):
    return "|".join(tokens_dict[tokno] for tokno in tokens_list)

In [36]:
def stringifyTokensTensor(tokens_tensor, token_boundaries=False) -> str:
    token_separator = "" if token_boundaries == False else "|"
    return token_separator.join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_tensor.tolist()).strip()

In [37]:
def stringifyAllignedTokensTensorPair(tokens_tensor1, tokens_tensor2, token_boundaries=False) -> str:

    def reddenString(txt, omit=False):
        if omit:
            return txt
        else:
            return Fore.RED+txt+Style.RESET_ALL
    
    
    token_separator = " " if token_boundaries == False else "|"

    output_len = max(len(tokens_tensor1), len(tokens_tensor2))

    str1 = ""
    str2 = ""

    for i in range(output_len):
        token1 = tokens_dict[tokens_tensor1.tolist()[i]]
        token2 = tokens_dict[tokens_tensor2.tolist()[i]]

        padded_length = max(len(token1), len(token2))

        omit = token1 == token2
            
        

        str1 += reddenString(token1.ljust(padded_length), omit) + token_separator
        str2 += reddenString(token2.ljust(padded_length), omit) + token_separator

    return str1.strip(), str2.strip()

In [38]:
def sortedListFind(sorted_list, sought_after_value) -> int:
    'Locate the leftmost value exactly equal to x'
    i = bisect.bisect_left(sorted_list, sought_after_value)
    if i != len(sorted_list) and sorted_list[i] == sought_after_value:
        return i
    else:
        return -1

In [39]:
tokens_dict = {}
tokens_dict_reversed = {}
char_dict = {}
char_dict_reversed = {}
with open("../orthography_cleaning/bpe_token_indices.csv", "r") as bpe_token_indices_file:
    with mmap.mmap(bpe_token_indices_file.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_bpe_file:
        line_no = 0
        for bin_line in iter(mmap_bpe_file.readline, b""):
            line = bin_line.decode("utf-8").strip()
            split_line = line.split(",")
            tokens_dict[int(split_line[0])] = split_line[1].strip()
            tokens_dict_reversed[split_line[1]] = int(split_line[0])
            if line_no < char_vocab_size:
                char_dict[int(split_line[0])] = split_line[1].strip()
                char_dict_reversed[split_line[1]] = int(split_line[0])
            line_no += 1
            if line_no == token_vocabulary_size:
                break

Validation subtexts: Xrabr: (none), Euch Sin: Prayer lying in a bed (1248, 1287), KF: Mass about the martyrs (2478, 2549), Mar: Luke 1 (28999, 30078), Psal: Psalm 43 (69524, 69861), Supr: Story of the soldier + Paul the simple (118212, 119284) Vita of Alexander, presbyter of Side (116350, 118211),  Vita Constantini: 1 (190722, 190901), Vita Methodii: none, Zogr: Matt 24 (211485, 211951), Assem: Mark 2 (268862, 269208), Savv: John 21 (322867, 323346), Pandects: (324010, 325048), Usp Sbor: Feodosij (395372, 395871), Yagichev: (415636, 416846), Kratuk Zlatostruy: (572508, 574487)

In [40]:
validation_offsets = [(1248, 1287), (2478, 2549), (28999, 30078), (69524, 69861), (116350, 118211), (190722, 190901), (211485, 211951), (268862, 269208), (322867, 323346), (324010, 325048), (395372, 395871), (415636, 416846), (572508, 574487)]

val_wordoffsets_list = []
val_text_wordoffsets = [0]
for tupl in validation_offsets:
    val_wordoffsets_list += range(tupl[0], tupl[1]+1)
    val_text_lngth = tupl[1] + 1 - tupl[0]
    val_text_wordoffsets.append(val_text_wordoffsets[-1] + val_text_lngth)
val_wordoffsets = set(val_wordoffsets_list)
val_text_wordoffsets = torch.tensor(val_text_wordoffsets[0:-1])
val_text_wordoffsets

tensor([   0,   40,  112, 1192, 1530, 3392, 3572, 4039, 4386, 4866, 5905, 6405,
        7616])

In [41]:
words_list = []
train_words_list = []
train_text_wordoffsets = []
val_words_list = []
with open("../orthography_cleaning/total_BERT_corpus.csv", "r") as BERT_words_file:
    current_text_id = 0
    wordoffset = 0
    val_wordoffset = 0
    train_wordoffset = 0
    for row in csv.DictReader(BERT_words_file, delimiter="|"):
        text_id = int(row["text_id"])
        if text_id != current_text_id:
            current_text_id = text_id
            if wordoffset not in val_wordoffsets:
                train_text_wordoffsets.append(train_wordoffset)                
            
        word = row["deep_cleaned"]
        if wordoffset in val_wordoffsets:           
            val_words_list.append(removeOOVChars(word))
            val_wordoffset += 1
        else:
            train_words_list.append(removeOOVChars(word))
            train_wordoffset += 1  
        wordoffset += 1
            
train_text_wordoffsets = torch.tensor(train_text_wordoffsets)
len(train_words_list) + len(val_words_list)

574488

In [42]:
len(val_words_list)

9596

In [43]:
# val_windows = getTokenWindows(torch.from_numpy(val_flat_token_ids).long(), torch.from_numpy(val_word_tokenoffsets)[val_text_wordoffsets])
# train_windows = getTokenWindows(torch.from_numpy(train_flat_token_ids).long(), torch.from_numpy(train_word_tokenoffsets)[train_text_wordoffsets])

val_windows = getTokenisedWindows(val_words_list, val_text_wordoffsets, 0.0)
train_windows = getTokenisedWindows(train_words_list, train_text_wordoffsets, 0.1)

In [44]:
val_windows.shape, train_windows.shape, (val_windows != 0).numel(), (train_windows != 0).numel()

(torch.Size([274, 96]), torch.Size([19803, 96]), 26304, 1901088)

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [46]:
print(device)

cpu


In [388]:
class OcsBertModel(torch.nn.Module):
    def __init__(self, token_vocab_size=token_vocabulary_size, token_embedding_dim=token_embedding_dim, token_seq_length=transformer_window_length, attention_heads=4, trans_layers=4, char_vocab_size=38, char_embedding_dim=32, lstm_hidden_size=64, lstm_layers=1, tag_slots_num=11, tag_slot_embedding_dim=32, decoder_lstm_hidden_size=256):
        super().__init__()
       
        self.token_embedder = torch.nn.Embedding(num_embeddings=token_vocab_size, embedding_dim=token_embedding_dim, padding_idx=0)
        self.positional_embedder = torch.nn.Embedding(num_embeddings=token_seq_length, embedding_dim=token_embedding_dim)
        transformer_encoder_layer = torch.nn.TransformerEncoderLayer(d_model=token_embedding_dim, nhead=attention_heads, dim_feedforward=4*token_embedding_dim, batch_first=True)
        self.transformer = torch.nn.TransformerEncoder(transformer_encoder_layer, trans_layers)

        self.token_linear_classifier_head = torch.nn.Linear(token_embedding_dim, token_vocabulary_size)
        
        
        self.token_embedding_dim = token_embedding_dim
        self.token_vocab_size = token_vocab_size
        self.MASK_token_idx = 2
        self.register_buffer("position_ids", torch.arange(token_seq_length).unsqueeze_(0))

    def corrupt_tokens(self, token_windows: torch.Tensor, padding_mask: torch.Tensor):
        B = token_windows.shape[0]
        W = token_windows.shape[1]
        flat_unpadded_token_windows = token_windows[~padding_mask] #(B, W)[(B, W)] -> (N) #~ is bitwise-NOT operator to flip all the mask bits
        N = flat_unpadded_token_windows.shape[0]

        flat_corrupted_token_windows = flat_unpadded_token_windows.clone()

        flat_corruption_mask = token_windows.new_zeros(N, dtype=torch.bool)
        percent_of_indices = torch.randperm(N, device=token_windows.device)[:int(N*0.15)] #.randperm(N) generates random indices between 0 and N-1, then we select the first 15% of them
        flat_corruption_mask[percent_of_indices] = True

        P = percent_of_indices.shape[0]
        r = torch.rand(P, device=token_windows.device)
        MASK_indices = percent_of_indices[r < 0.8]
        other_token_indices = percent_of_indices[(r >= 0.8) & (r < 0.9)] #has to be bitwise AND because (r < 0.9) produces a tensor 

        flat_corrupted_token_windows[MASK_indices] = self.MASK_token_idx
        flat_corrupted_token_windows[other_token_indices] = torch.randint(3, self.token_vocab_size, (other_token_indices.shape[0],), device=token_windows.device)

        flat_padding_mask = padding_mask.reshape(B*W)

        flattened_repadded_corrupted_token_windows = flat_corrupted_token_windows.new_zeros(B*W)
        flattened_repadded_corrupted_token_windows[~flat_padding_mask] = flat_corrupted_token_windows

        corrupted_token_windows = flattened_repadded_corrupted_token_windows.reshape(B, W)

        return corrupted_token_windows, flat_corruption_mask
        #return token_windows, flat_corruption_mask #test to see if it can predict non-corrupted tokens

    
    def forward(self, token_windows, test_corruption, test_corruption_mask) -> torch.Tensor:

        padding_mask = (token_windows == 0) #(B, W)
        # corrupted_tokens_windows, flat_corruption_mask = self.corrupt_tokens(token_windows, padding_mask) #(B, W)
        corrupted_tokens_windows, flat_corruption_mask = test_corruption, test_corruption_mask

        ### TRANSFORMER ###
        token_embeddings = self.token_embedder(corrupted_tokens_windows) #(B, W, 256)
        positional_embeddings = self.positional_embedder(self.position_ids.expand(token_embeddings.size(0), -1))
        
        token_states = self.transformer(token_embeddings + positional_embeddings, src_key_padding_mask=padding_mask) #(B, W, 256)

        ###MASKED TOKEN PREDICTION###
        flat_corrupted_tokens = token_states[~padding_mask][flat_corruption_mask] #(Nmasked, 256)

        corrupted_token_logits = self.token_linear_classifier_head(flat_corrupted_tokens) #(Nmasked, token_vocab_size)

        return corrupted_token_logits, token_windows[~padding_mask][flat_corruption_mask]

In [72]:
class tokenWindowsDataset(torch.utils.data.Dataset):
    def __init__(self, token_windows):
        self.token_windows = token_windows

    def __len__(self):
        return self.token_windows.shape[0]

    def __getitem__(self, idx):
        
        return self.token_windows[idx]

In [273]:
train_dataloader = getTokenWindowsDataLoader(train_windows, 32, True)
val_dataloader = getTokenWindowsDataLoader(val_windows, 32, False)

In [ ]:
network = OcsBertModel()
cross_entropy_loss = torch.nn.modules.loss.CrossEntropyLoss()
adamw_optimiser = torch.optim.AdamW(network.parameters(), lr=1e-4)
network.to(device)
#network.load_state_dict(torch.load("MorphologyLSTMTransformerModel_POS_only_18epochs.pt", map_location=device))

OcsBertModel(
  (token_embedder): Embedding(8043, 256, padding_idx=0)
  (positional_embedder): Embedding(96, 256)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (token_linear_classifier_head): Linear(in_features=256, out_features=8043, bias=True)
)

In [ ]:
network.train()
train_dataloader_iter = iter(getTokenWindowsDataLoader(train_windows, 32, True))
batch = next(train_dataloader_iter)

In [432]:
# try:
#     batch = next(train_dataloader_iter)
# except StopIteration:
#     print("Reached end of dataset")

network.train()   

corrupted_token_logits, correct_tokens = network(test_batch, test_corruption, test_corruption_mask)

loss = cross_entropy_loss(corrupted_token_logits, correct_tokens)

adamw_optimiser.zero_grad()
loss.backward()
adamw_optimiser.step()

print("Loss:", loss.item())

# pred_tokens, correct_tokens = stringifyAllignedTokensTensorPair(torch.argmax(corrupted_token_logits, dim=1)[:20], correct_tokens[:20], True)

# print(pred_tokens, correct_tokens, sep="\n")

# stringifyTokensTensor(torch.argmax(sft_mx(corrupted_token_logits), dim=1)[:20], True), stringifyTokensTensor(correct_tokens[:20], True)

Loss: 0.08417995274066925


In [51]:
training_loss_values = []
batches_counts = []
training_epochs = []
per_epoch_loss_values = []
validation_average_per_epoch_loss_values = []

In [52]:
val_windows = getTokenisedWindows(val_words_list, val_text_wordoffsets, 0.0)
val_dataloader_iter = iter(getTokenWindowsDataLoader(val_windows, 32, False))

In [ ]:
network.train()
total_epoch_number = 30

batches_count = 0
best_validation_loss = 99999.9
for epoch_no in range(total_epoch_number):

    train_windows = getTokenisedWindows(train_words_list, train_text_wordoffsets, 0.1)
    train_dataloader_iter = iter(getTokenWindowsDataLoader(train_windows, 32, True))
    
    network.train()

    epoch_loss_values = []
    for batch in train_dataloader_iter:
    
        corrupted_token_logits, correct_tokens = network(batch)
        loss = cross_entropy_loss(corrupted_token_logits, correct_tokens)

        adamw_optimiser.zero_grad()
        loss.backward()
        adamw_optimiser.step()
        
        training_loss_values.append(loss.item())
        epoch_loss_values.append(loss.detach())
        
        batches_counts.append(batches_count)
        batches_count += 1
    
    epoch_training_loss = torch.stack(epoch_loss_values).mean().item()
    per_epoch_loss_values.append(epoch_training_loss)

    network.eval()
    validation_per_window_loss_values = []
    validation_per_window_tagslot_loss_values = []
    with torch.inference_mode():
        for batch in val_dataloader_iter:

            corrupted_token_logits, correct_tokens = network(batch)
            loss = cross_entropy_loss(corrupted_token_logits, correct_tokens)
            
            validation_per_window_loss_values.append(loss)

        mean_validation_per_batch_loss = torch.stack(validation_per_window_loss_values).mean().item()
        validation_average_per_epoch_loss_values.append(mean_validation_per_batch_loss)

        if epoch_no > 9:
          print(f"Epoch no. {epoch_no}\n    Training loss: {epoch_training_loss}\n    Validation loss: {mean_validation_per_batch_loss}")
          if mean_validation_per_batch_loss < best_validation_loss:
            torch.save(obj=network.state_dict(), f="OCSBertModel_bestcheckpoint.pt")
            print(f"Saving model at epoch no. {epoch_no} because its validation loss was lower than the previous best validation-loss of {best_validation_loss}")
            best_validation_loss = mean_validation_per_batch_loss
    training_epochs.append(epoch_no)
torch.save(obj=network.state_dict(), f=f"OCSBertModel_bestcheckpoint_{total_epoch_number}epochs.pt")

In [ ]:
validation_average_per_epoch_loss_values, per_epoch_loss_values

In [ ]:
plt.plot(batches_counts, training_loss_values, label="training loss")
plt.ylabel("Loss")
plt.xlabel("batches processed")
plt.legend()
print(training_loss_values[batches_counts[-1]])
plt.show()

In [ ]:
plt.plot(training_epochs, validation_average_per_epoch_loss_values)

In [ ]:
plt.plot(training_epochs, per_epoch_loss_values)